# 规则引擎示例

本notebook展示如何使用hscredit的规则引擎功能，包括规则定义、逻辑运算、表达式优化和效果评估。

In [1]:
import sys
sys.path.insert(0, '/Users/xiaoxi/CodeBuddy/hscredit/hscredit')


import pandas as pd
import numpy as np
from hscredit import Rule, optimize_expr, beautify_expr, get_expr_variables

## 1. 加载数据

In [2]:
# 加载示例数据
df = pd.read_excel('/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx')
print(f"数据形状: {df.shape}")
print(f"列名: {df.columns.tolist()}")

# 创建目标变量: MOB1 > 15 表示账龄超过15个月
df['target'] = (df['MOB1'] > 15).astype(int)
print(f"\n目标变量分布:")
print(df['target'].value_counts())

数据形状: (22729, 12)
列名: ['MOB1', 'MOB2', '青云24', '游昆定制分80', '百融定制分V9', '中智小牛分C3', '度小满欺诈因子V6PRO多头版', '百行百川分FPV1', '设备黑名单', '放款期数']

目标变量分布:
target
0    21122
1     1607
Name: count, dtype: int64


## 2. 定义基础规则

In [3]:
# 使用数据中的数值型变量创建规则
# 规则表达式使用pandas eval语法

# 规则1: 青云24分数大于700
rule_qy24 = Rule('青云24 > 700')

# 规则2: 游昆定制分大于750
rule_yk_score = Rule('游昆定制分80 > 750')

# 规则3: 百融定制分大于某个阈值
rule_br_score = Rule('百融定制分V9 > 650')

# 规则4: 设备黑名单次数大于0
rule_blacklist = Rule('设备黑名单 > 0')

print("基础规则定义:")
print(f"rule_qy24: {rule_qy24.expr}")
print(f"rule_yk_score: {rule_yk_score.expr}")
print(f"rule_br_score: {rule_br_score.expr}")
print(f"rule_blacklist: {rule_blacklist.expr}")

基础规则定义:
rule_qy24: 青云24 > 700
rule_yk_score: 游昆定制分80 > 750
rule_br_score: 百融定制分V9 > 650
rule_blacklist: 设备黑名单 > 0


## 3. 规则逻辑运算与表达式优化

In [4]:
print("=" * 60)
print("3.1 幂等律 (Idempotent Law): A & A = A, A | A = A")
print("=" * 60)
r1 = Rule('青云24 > 700')
result = r1 & r1
print(f"r1 & r1: {result.expr}")

result = r1 | r1
print(f"r1 | r1: {result.expr}")

3.1 幂等律 (Idempotent Law): A & A = A, A | A = A
r1 & r1: 青云24 > 700
r1 | r1: 青云24 > 700


In [5]:
print("=" * 60)
print("3.2 双重否定 (Double Negation): ~~A = A")
print("=" * 60)
r1 = Rule('青云24 > 700')
result = ~~r1
print(f"~~r1: {result.expr}")

3.2 双重否定 (Double Negation): ~~A = A
~~r1: 青云24 > 700


In [6]:
print("=" * 60)
print("3.3 吸收率 (Absorption Law): A | (A & B) = A")
print("=" * 60)
r1 = Rule('青云24 > 700')
r2 = Rule('游昆定制分80 > 750')

# A | (A & B) = A
result = r1 | (r1 & r2)
print(f"r1 | (r1 & r2): {result.expr}")

# (A & B) | B = B
result = (r1 & r2) | r2
print(f"(r1 & r2) | r2: {result.expr}")

# A & (A | B) = A
result = r1 & (r1 | r2)
print(f"r1 & (r1 | r2): {result.expr}")

3.3 吸收率 (Absorption Law): A | (A & B) = A
r1 | (r1 & r2): 青云24 > 700 | (青云24 > 700 & 游昆定制分80 > 750)
(r1 & r2) | r2: 游昆定制分80 > 750
r1 & (r1 | r2): 青云24 > 700 | 游昆定制分80 > 750


In [7]:
print("=" * 60)
print("3.4 基本逻辑运算")
print("=" * 60)
r1 = Rule('青云24 > 700')
r2 = Rule('游昆定制分80 > 750')

# 与运算 (AND)
result = r1 & r2
print(f"r1 & r2 (与): {result.expr}")

# 或运算 (OR)
result = r1 | r2
print(f"r1 | r2 (或): {result.expr}")

# 异或运算 (XOR)
result = r1 ^ r2
print(f"r1 ^ r2 (异或): {result.expr}")

# 非运算 (NOT)
result = ~r1
print(f"~r1 (非): {result.expr}")

3.4 基本逻辑运算
r1 & r2 (与): 青云24 > 700 & 游昆定制分80 > 750
r1 | r2 (或): 青云24 > 700 | 游昆定制分80 > 750
r1 ^ r2 (异或): 青云24 > 700 ^ 游昆定制分80 > 750
~r1 (非): ~(青云24 > 700)


In [8]:
print("=" * 60)
print("3.5 复杂组合规则")
print("=" * 60)
r1 = Rule('青云24 > 700')
r2 = Rule('游昆定制分80 > 750')
r3 = Rule('百融定制分V9 > 650')
r4 = Rule('设备黑名单 > 0')

# 组合: (规则1 与 规则2) 或 规则3
complex_rule = (r1 & r2) | r3
print(f"(r1 & r2) | r3: {complex_rule.expr}")

# 更复杂的组合
complex_rule = (r1 & r2) | (r3 & ~r4)
print(f"(r1 & r2) | (r3 & ~r4): {complex_rule.expr}")

3.5 复杂组合规则
(r1 & r2) | r3: (青云24 > 700 & 游昆定制分80 > 750) | 百融定制分V9 > 650
(r1 & r2) | (r3 & ~r4): (青云24 > 700 & 游昆定制分80 > 750) | (百融定制分V9 > 650 & ~(设备黑名单 > 0))


## 4. 优化器函数

In [9]:
# 优化表达式: 应用布尔代数定律简化
print("=" * 60)
print("4.1 optimize_expr - 简化表达式")
print("=" * 60)

expr1 = '(青云24 > 700) & (青云24 > 700)'
print(f"优化前: {expr1}")
print(f"优化后: {optimize_expr(expr1)}")

expr2 = '~~(青云24 > 700)'
print(f"\n优化前: {expr2}")
print(f"优化后: {optimize_expr(expr2)}")

expr3 = '(青云24 > 700 & 游昆定制分80 > 750) | 游昆定制分80 > 750'
print(f"\n优化前: {expr3}")
print(f"优化后: {optimize_expr(expr3)}")

4.1 optimize_expr - 简化表达式
优化前: (青云24 > 700) & (青云24 > 700)
优化后: 青云24 > 700

优化前: ~~(青云24 > 700)
优化后: 青云24 > 700

优化前: (青云24 > 700 & 游昆定制分80 > 750) | 游昆定制分80 > 750
优化后: 青云24 > 700 & 游昆定制分80 > 750 | 游昆定制分80 > 750


In [10]:
print("=" * 60)
print("4.2 beautify_expr - 美化表达式格式")
print("=" * 60)

expr = '(青云24 > 700) & (游昆定制分80 > 750)'
print(f"美化前: {expr}")
print(f"美化后: {beautify_expr(expr)}")

4.2 beautify_expr - 美化表达式格式
美化前: (青云24 > 700) & (游昆定制分80 > 750)
美化后: 青云24 > 700 & 游昆定制分80 > 750


In [11]:
print("=" * 60)
print("4.3 get_expr_variables - 提取变量名")
print("=" * 60)

expr = '(青云24 > 700) & (游昆定制分80 > 750)'
print(f"表达式: {expr}")
print(f"变量: {get_expr_variables(expr)}")

4.3 get_expr_variables - 提取变量名
表达式: (青云24 > 700) & (游昆定制分80 > 750)
变量: []


## 5. 规则效果评估

In [12]:
# 使用Rule的report方法评估规则效果
print("=" * 60)
print("5.1 单规则效果评估")
print("=" * 60)

# 定义规则: 青云24分数大于700
rule1 = Rule('青云24 > 700')

# 生成规则报告
report = rule1.report(df, target='target', desc='青云24分数>700')
print("规则报告:")
report

5.1 单规则效果评估
规则报告:


,规则分类,指标名称,指标含义,分箱,样本总数,样本占比,好样本数,好样本占比,坏样本数,坏样本占比,坏样本率,LIFT值,坏账改善,风险拒绝比,准确率,精确率,召回率,F1分数
0,验证规则,青云24 > 700,青云24分数>700,命中,1550,0.068195,1497,0.065863,53,0.002332,0.034194,0.483625,-0.035214,-0.516375,0.865766,0.034194,0.032981,0.033576
1,验证规则,青云24 > 700,青云24分数>700,未命中,21179,0.931805,19625,0.863434,1554,0.068371,0.073375,1.037791,0.035214,0.037791,0.134234,0.073375,0.967019,0.136400


In [13]:
# 定义规则: 游昆定制分大于750
rule2 = Rule('游昆定制分80 > 750')

report2 = rule2.report(df, target='target', desc='游昆定制分>750')
print("规则报告:")
print(report2)

规则报告:
   规则分类           指标名称       指标含义   分箱   样本总数  样本占比   好样本数     好样本占比  坏样本数  \
0  验证规则  游昆定制分80 > 750  游昆定制分>750   命中      0   0.0      0  0.000000     0   
1  验证规则  游昆定制分80 > 750  游昆定制分>750  未命中  22729   1.0  21122  0.929297  1607   

      坏样本占比      坏样本率  LIFT值  坏账改善  风险拒绝比       准确率       精确率  召回率      F1分数  
0  0.000000  0.000000    0.0   0.0    NaN  0.929297  0.000000  0.0  0.000000  
1  0.070703  0.070703    1.0   0.0    0.0  0.070703  0.070703  1.0  0.132068  


In [14]:
print("=" * 60)
print("5.2 组合规则效果评估")
print("=" * 60)

# 组合规则: 青云24 > 700 或 游昆定制分 > 750
combined_rule = rule1 | rule2

report_combined = combined_rule.report(df, target='target', desc='青云24>700 或 游昆定制分>750')
print("组合规则报告:")
print(report_combined)

5.2 组合规则效果评估
组合规则报告:
   规则分类                        指标名称                  指标含义   分箱   样本总数  \
0  验证规则  青云24 > 700 | 游昆定制分80 > 750  青云24>700 或 游昆定制分>750   命中   1550   
1  验证规则  青云24 > 700 | 游昆定制分80 > 750  青云24>700 或 游昆定制分>750  未命中  21179   

       样本占比   好样本数     好样本占比  坏样本数     坏样本占比      坏样本率     LIFT值      坏账改善  \
0  0.068195   1497  0.065863    53  0.002332  0.034194  0.483625 -0.035214   
1  0.931805  19625  0.863434  1554  0.068371  0.073375  1.037791  0.035214   

      风险拒绝比       准确率       精确率       召回率      F1分数  
0 -0.516375  0.865766  0.034194  0.032981  0.033576  
1  0.037791  0.134234  0.073375  0.967019  0.136400  


In [15]:
# 组合规则: 青云24 > 700 且 游昆定制分 > 750
combined_rule_and = rule1 & rule2

report_and = combined_rule_and.report(df, target='target', desc='青云24>700 且 游昆定制分>750')
print("组合规则(与)报告:")
print(report_and)

组合规则(与)报告:
   规则分类                        指标名称                  指标含义   分箱   样本总数  样本占比  \
0  验证规则  青云24 > 700 & 游昆定制分80 > 750  青云24>700 且 游昆定制分>750   命中      0   0.0   
1  验证规则  青云24 > 700 & 游昆定制分80 > 750  青云24>700 且 游昆定制分>750  未命中  22729   1.0   

    好样本数     好样本占比  坏样本数     坏样本占比      坏样本率  LIFT值  坏账改善  风险拒绝比       准确率  \
0      0  0.000000     0  0.000000  0.000000    0.0   0.0    NaN  0.929297   
1  21122  0.929297  1607  0.070703  0.070703    1.0   0.0    0.0  0.070703   

        精确率  召回率      F1分数  
0  0.000000  0.0  0.000000  
1  0.070703  1.0  0.132068  


## 6. 使用overdue参数

In [16]:
# 使用overdue参数进行逾期分析
print("=" * 60)
print("6.1 使用MOB1作为逾期天数分析")
print("=" * 60)

# 使用MOB1作为逾期天数字段
rule_mob = Rule('青云24 > 700')

# 设置overdue参数: MOB1 > 15 为坏账
report_mob = rule_mob.report(df, overdue='MOB1', dpds=15, desc='青云24分数>700')
print("基于MOB1>15的规则报告:")
print(report_mob)

6.1 使用MOB1作为逾期天数分析
基于MOB1>15的规则报告:
   规则详情                                               MOB1 DPD15+            \
   规则分类        指标名称        指标含义   分箱   样本总数      样本占比        好样本数     好样本占比   
0  验证规则  青云24 > 700  青云24分数>700   命中   1550  0.068195        1497  0.065863   
1  验证规则  青云24 > 700  青云24分数>700  未命中  21179  0.931805       19625  0.863434   

                                                                               \
   坏样本数     坏样本占比      坏样本率     LIFT值      坏账改善     风险拒绝比       准确率       精确率   
0    53  0.002332  0.034194  0.483625 -0.035214 -0.516375  0.865766  0.034194   
1  1554  0.068371  0.073375  1.037791  0.035214  0.037791  0.134234  0.073375   

                       
        召回率      F1分数  
0  0.032981  0.033576  
1  0.967019  0.136400  


In [17]:
print("=" * 60)
print("6.2 使用MOB2作为逾期天数分析")
print("=" * 60)

report_mob2 = rule_mob.report(df, overdue=['MOB1', 'MOB2'], dpds=[0, 15], desc='青云24分数>700')
print("基于MOB2>15的规则报告:")
print(report_mob2)

6.2 使用MOB2作为逾期天数分析
基于MOB2>15的规则报告:
   规则详情                                               MOB1 DPD0+            \
   规则分类        指标名称        指标含义   分箱   样本总数      样本占比       好样本数     好样本占比   
0  验证规则  青云24 > 700  青云24分数>700   命中   1550  0.068195       1440  0.063355   
1  验证规则  青云24 > 700  青云24分数>700  未命中  21179  0.931805      17904  0.787716   

                   ... MOB2 DPD15+                                          \
   坏样本数     坏样本占比  ...        坏样本数     坏样本占比      坏样本率     LIFT值      坏账改善   
0   110  0.004840  ...          78  0.003432  0.050323  0.456600 -0.037057   
1  3275  0.144089  ...        2427  0.106780  0.114595  1.039769  0.037057   

                                                     
      风险拒绝比       准确率       精确率       召回率      F1分数  
0 -0.543400  0.828457  0.050323  0.031138  0.038471  
1  0.039769  0.171543  0.114595  0.968862  0.204948  

[2 rows x 54 columns]


## 7. 使用先验规则(prior_rules)进行规则置换分析

`prior_rules`参数用于规则置换场景，当已有先验规则时，评估新规则在**未命中先验规则样本**上的效果。

**典型应用场景**：
- 已有规则A（先验规则），想评估规则B在剩余样本上的效果
- 规则置换：评估新规则相比旧规则能额外拦截多少风险
- 规则叠加：在先验规则已拦截的样本外，评估新增规则的价值

In [18]:
print("=" * 60)
print("7.1 先验规则基础用法")
print("=" * 60)

# 定义先验规则: 青云24分数大于600
prior_rule = Rule('青云24 > 600')

# 定义待验证规则: 青云24分数大于700（更严格的规则）
new_rule = Rule('青云24 > 700')

# 不使用prior_rules: 在全量样本上评估
report_full = new_rule.report(df, target='target', desc='全量样本-青云24>700')
print("\n全量样本评估:")
print(report_full[['规则分类', '分箱', '样本总数', '样本占比', '坏样本率']])

# 使用prior_rules: 只在未命中先验规则的样本上评估
report_partial = new_rule.report(
    df, 
    target='target', 
    desc='剩余样本-青云24>700',
    prior_rules=prior_rule
)
print("\n先验规则后的剩余样本评估:")
print(report_partial[['规则分类', '分箱', '样本总数', '样本占比', '坏样本率']])

7.1 先验规则基础用法

全量样本评估:
   规则分类   分箱   样本总数      样本占比      坏样本率
0  验证规则   命中   1550  0.068195  0.034194
1  验证规则  未命中  21179  0.931805  0.073375

先验规则后的剩余样本评估:
   规则分类   分箱   样本总数      样本占比      坏样本率
0  先验规则   命中  11248  0.494874  0.058944
1  先验规则  未命中  11481  0.505126  0.082223
2  验证规则   命中      0  0.000000  0.000000
3  验证规则  未命中  11481  1.000000  0.082223


In [19]:
print("=" * 60)
print("7.2 规则置换效果分析")
print("=" * 60)

# 场景：已有规则 "百融定制分V9 > 500"，想评估换成 "百融定制分V9 > 650" 的效果

# 先验规则（旧规则）
old_rule = Rule('百融定制分V9 > 500')

# 新规则（更严格的阈值）
new_strict_rule = Rule('百融定制分V9 > 650')

# 生成对比报告
swap_report = new_strict_rule.report(
    df,
    target='target',
    desc='置换后规则-百融>650',
    prior_rules=old_rule,
    filter_cols=['规则分类', '分箱', '样本总数', '样本占比', '坏样本率', 'LIFT值', '坏账改善', '风险拒绝比']
)

print("规则置换分析报告:")
print(swap_report)

7.2 规则置换效果分析
规则置换分析报告:
   规则分类           指标名称          指标含义   分箱   样本总数      样本占比      坏样本率  \
0  先验规则  百融定制分V9 > 500  置换后规则-百融>650   命中  22682  0.997932  0.070585   
1  先验规则  百融定制分V9 > 500  置换后规则-百融>650  未命中     47  0.002068  0.127660   
2  验证规则  百融定制分V9 > 650  置换后规则-百融>650   命中      0  0.000000  0.000000   
3  验证规则  百融定制分V9 > 650  置换后规则-百融>650  未命中     47  1.000000  0.127660   

      LIFT值      坏账改善     风险拒绝比  
0  0.998331 -0.001666 -0.001669  
1  1.805585  0.001666  0.805585  
2  0.000000  0.000000       NaN  
3  1.000000  0.000000  0.000000  


In [20]:
print("=" * 60)
print("7.3 多规则叠加效果")
print("=" * 60)

# 场景：已有两条规则，想评估增加第三条规则的效果

# 先验规则组合
existing_rules = Rule('青云24 > 600') | Rule('百融定制分V9 > 500')

# 新增规则
additional_rule = Rule('设备黑名单 > 0')

# 评估新增规则在已有规则剩余样本上的效果
addition_report = additional_rule.report(
    df,
    target='target',
    desc='设备黑名单规则',
    prior_rules=existing_rules,
    filter_cols=['规则分类', '分箱', '样本总数', '样本占比', '坏样本率', '坏账改善']
)

print("新增规则效果分析:")
print(addition_report)

7.3 多规则叠加效果
新增规则效果分析:
   规则分类                        指标名称     指标含义   分箱   样本总数      样本占比      坏样本率  \
0  先验规则  青云24 > 600 | 百融定制分V9 > 500  设备黑名单规则   命中  22690  0.998284  0.070692   
1  先验规则  青云24 > 600 | 百融定制分V9 > 500  设备黑名单规则  未命中     39  0.001716  0.076923   
2  验证规则                 设备黑名单 > 0  设备黑名单规则   命中     25  0.641026  0.040000   
3  验证规则                 设备黑名单 > 0  设备黑名单规则  未命中     14  0.358974  0.142857   

       坏账改善  
0 -0.000151  
1  0.000151  
2 -0.307692  
3  0.307692  


In [21]:
print("=" * 60)
print("7.4 prior_rules与overdue参数联合使用")
print("=" * 60)

# 先验规则
prior = Rule('青云24 > 600')

# 验证规则
test_rule = Rule('游昆定制分80 > 700')

# 结合overdue参数进行多维度逾期分析
multi_dpd_report = test_rule.report(
    df,
    overdue=['MOB1', 'MOB2'],
    dpds=[15, 30],
    prior_rules=prior,
    desc='游昆分>700'
)

print("多逾期标签分析（含先验规则）:")
print(multi_dpd_report.head())

7.4 prior_rules与overdue参数联合使用
多逾期标签分析（含先验规则）:
   规则详情                                               MOB1 DPD15+            \
   规则分类           指标名称     指标含义   分箱   样本总数      样本占比        好样本数     好样本占比   
0  先验规则     青云24 > 600  游昆分>700   命中  11248  0.494874       10585  0.465705   
1  先验规则     青云24 > 600  游昆分>700  未命中  11481  0.505126       10537  0.463593   
2  验证规则  游昆定制分80 > 700  游昆分>700   命中   7393  0.643933        6818  0.593851   
3  验证规则  游昆定制分80 > 700  游昆分>700  未命中   4088  0.356067        3719  0.323926   

                  ... MOB2 DPD30+                                          \
  坏样本数     坏样本占比  ...        坏样本数     坏样本占比      坏样本率     LIFT值      坏账改善   
0  663  0.029170  ...         655  0.028818  0.058233  0.796850 -0.100534   
1  944  0.041533  ...        1006  0.044261  0.087623  1.199027  0.100534   
2  575  0.050083  ...         597  0.051999  0.080752  0.921585 -0.050494   
3  369  0.032140  ...         409  0.035624  0.100049  1.141811  0.050494   

                

In [22]:
print("=" * 60)
print("7.5 规则置换决策示例")
print("=" * 60)

# 实际业务场景：评估规则置换的收益

print("场景：将'百融>500'置换为'百融>650'\n")

# 步骤1: 原规则效果
original_rule = Rule('百融定制分V9 > 500')
original_report = original_rule.report(df, target='target', desc='原规则')
original_hit = original_report[original_report['分箱'] == '命中']['样本总数'].values[0]
original_bad_rate = original_report[original_report['分箱'] == '命中']['坏样本率'].values[0]

print(f"原规则(百融>500)拦截样本: {original_hit}")
print(f"原规则拦截样本坏样本率: {original_bad_rate:.4f}")

# 步骤2: 新规则在剩余样本上的效果
new_rule = Rule('百融定制分V9 > 650')
swap_report = new_rule.report(df, target='target', desc='新规则', prior_rules=original_rule)

# 提取关键指标
prior_hit = swap_report[swap_report['规则分类'] == '先验规则']['样本总数'].values[0]
new_hit = swap_report[(swap_report['规则分类'] == '验证规则') & (swap_report['分箱'] == '命中')]['样本总数'].values[0]

print(f"\n先验规则命中样本: {prior_hit}")
print(f"置换后新增命中样本: {new_hit}")
print(f"置换后总拦截样本: {prior_hit + new_hit}")

# 计算置换效果
if new_hit > 0:
    new_bad_rate = swap_report[(swap_report['规则分类'] == '验证规则') & (swap_report['分箱'] == '命中')]['坏样本率'].values[0]
    print(f"\n新拦截样本坏样本率: {new_bad_rate:.4f}")
    print(f"\n置换决策建议:")
    if new_bad_rate > original_bad_rate * 1.5:
        print("  ✅ 建议置换 - 新规则拦截样本的坏样本率显著高于原规则")
    elif new_bad_rate > original_bad_rate:
        print("  ⚠️ 可考虑置换 - 新规则拦截样本的坏样本率有所提升")
    else:
        print("  ❌ 不建议置换 - 新规则拦截样本的坏样本率未提升")

7.5 规则置换决策示例
场景：将'百融>500'置换为'百融>650'

原规则(百融>500)拦截样本: 22682
原规则拦截样本坏样本率: 0.0706

先验规则命中样本: 22682
置换后新增命中样本: 0
置换后总拦截样本: 22682


## 8. 规则集评估

In [23]:
from hscredit import ruleset_report

# 定义多个规则
rules = [
    Rule('青云24 > 700'),
    Rule('游昆定制分80 > 750'),
    Rule('百融定制分V9 > 650'),
]

# 生成规则集报告
print("=" * 60)
print("7. 规则集报告")
print("=" * 60)

ruleset = ruleset_report(df, rules, overdue=['MOB1', 'MOB2'], dpds=15)
print("规则集报告:")
ruleset

7. 规则集报告
规则集报告:


MOB1 DPD15+           规则详情                  MOB1 DPD15+                  \
         指标含义             分箱   样本总数      样本占比        好样本数     好样本占比  坏样本数   
0                       原始样本  22729  1.000000       21122  0.929297  1607   
1                 青云24 > 700   1550  0.068195        1497  0.065863    53   
2                       剩余样本  21179  0.931805       19625  0.863434  1554   
3              游昆定制分80 > 750      0  0.000000           0  0.000000     0   
4                       剩余样本  21179  1.000000       19625  0.926625  1554   
5              百融定制分V9 > 650  18180  0.858397       16975  0.801501  1205   
6                       剩余样本   2999  0.141603        2650  0.125124   349   
7                       所有规则  19730  0.868054       18472  0.812706  1258   

                                 ... MOB2 DPD15+                      \
      坏样本占比      坏样本率     LIFT值  ...        坏样本数     坏样本占比      坏样本率   
0  0.070703  0.070703  1.000000  ...        2505  0.110212  0.110212   
1  0.002332  0.034194  0.483625  ...          78  0.003432  0.050323   
2  0.068371  0.073375  1.037791  ...        2427  0.106780  0.114595   
3  0.000000  0.000000  0.000000  ...           0  0.000000  0.000000   
4  0.073375  0.073375  1.000000  ...        2427  0.114595  0.114595   
5  0.056896  0.066282  0.903332  ...        1901  0.089759  0.104565   
6  0.016479  0.116372  1.586001  ...         526  0.024836  0.175392   
7  0.055348  0.063761  0.901816  ...        1979  0.087069  0.100304   

                                                                         
      LIFT值      坏账改善     风险拒绝比       准确率       精确率       召回率      F1分数  
0  1.000000  0.000000  0.000000  0.195873  0.100304  0.790020  0.178008  
1  0.456600 -0.037057 -0.543400  0.828457  0.050323  0.031138  0.038471  
2  1.039769  0.037057  0.039769  0.171543  0.114595  0.968862  0.204948  
3  0.000000  0.000000       NaN  0.885405  0.000000  0.000000  0.000000  
4  1.000000  0.000000  0.000000  0.114595  0.114595  1.000000  0.205626  
5  0.912481 -0.075126 -0.087519  0.206525  0.104565  0.783272  0.184500  
6  1.530541  0.075126  0.530541  0.793475  0.175392  0.216728  0.193881  
7  0.910105 -0.078034 -0.089895  0.195873  0.100304  0.790020  0.178008  

[8 rows x 29 columns]

In [24]:
ruleset = ruleset_report(df, rules, target='target', filter_cols=['样本总数', '样本占比', '坏样本率'])
print("规则集报告:")
ruleset

规则集报告:


,指标含义,分箱,样本总数,样本占比,坏样本率
0,,原始样本,22729,1.000000,0.070703
1,,青云24 > 700,1550,0.068195,0.034194
2,,剩余样本,21179,0.931805,0.073375
3,,游昆定制分80 > 750,0,0.000000,0.000000
4,,剩余样本,21179,1.000000,0.073375
5,,百融定制分V9 > 650,18180,0.858397,0.066282
6,,剩余样本,2999,0.141603,0.116372
7,,所有规则,19730,0.868054,0.063761


## 9. 规则预测

In [25]:
# 使用规则进行预测
print("=" * 60)
print("8. 规则预测")
print("=" * 60)

rule = Rule('青云24 > 700')

# 预测: 返回布尔Series
predictions = rule.predict(df)
print(f"预测结果类型: {type(predictions)}")
print(f"命中样本数: {predictions.sum()}")
print(f"未命中样本数: {(~predictions).sum()}")

# 过滤数据
matched_df = rule.filter(df)
print(f"\n规则命中的数据形状: {matched_df.shape}")

8. 规则预测
预测结果类型: <class 'pandas.core.series.Series'>
命中样本数: 1550
未命中样本数: 21179

规则命中的数据形状: (1550, 13)


## 10. 总结

### 规则引擎主要功能:

1. **规则定义**: 使用`Rule(expr)`定义规则表达式

2. **逻辑运算**: 支持`&` (与), `|` (或), `^` (异或), `~` (非)

3. **表达式优化**: 
   - `optimize_expr()`: 应用布尔代数定律简化表达式
   - `beautify_expr()`: 美化表达式格式
   - `get_expr_variables()`: 提取变量名

4. **规则评估**: 
   - `Rule.report()`: 单规则效果评估
   - `ruleset_report()`: 多规则综合评估

5. **规则预测**: 
   - `Rule.predict()`: 应用规则预测
   - `Rule.filter()`: 根据规则过滤数据